# Day 08 — Chạy model pre-label trên Google Colab (cho máy yếu)

**Dùng khi:** laptop không GPU hoặc quá chậm. Colab cho GPU miễn phí (T4) để **chạy model** (Part 2–5). **CVAT, review, đo số, viết báo cáo vẫn làm trên máy bạn.**

**Luồng:** Colab chạy `prelabel.py` → tải `colab_outputs.zip` về → nạp file nhãn vào CVAT (cách A, `docs/CVAT_AI.md`) → review → các script đo (`review_stats.py`...) chạy trên máy bạn (nhẹ, không cần GPU).

> Colab **không** nối được tới CVAT `localhost` của bạn: ta chỉ lấy file nhãn từ Colab. Bài vẫn là **cá nhân**: commit từ repo trên máy bạn.

**Trước khi chạy:** menu **Runtime → Change runtime type → T4 GPU**. Chạy các ô từ trên xuống.

**Bản chuẩn bị cho Trần Xuân Bắc — 2A202603132.** Chưa chạy model. Máy local dùng môi trường nhẹ và CVAT Docker. Hoàn tất gán tay, lưu và export 4 ảnh gold trước khi chạy Part 2.


**Repo dùng trong notebook:** `trxuanbac/K5-L23-DAY08-2A202603132-TranXuanBac-AIPrelabel`, nhánh `main`.

**Chạy theo chặng:** mục 0–5 → tải kết quả → quét ngưỡng trên máy → nhập ngưỡng ở mục 6. Mục 7 là ví dụ cho một vật; cần thực hiện đủ 3 vật theo bài. Khi Colab chưa cấp runtime, các ô chưa thể chạy.


## 0. Kiểm tra GPU và lấy mã nguồn

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "-L"], check=False)
else:
    print("Chưa có GPU: có thể chạy CPU nếu Colab cấp được runtime.")

REPO_URL = "https://github.com/trxuanbac/K5-L23-DAY08-2A202603132-TranXuanBac-AIPrelabel.git"
REPO_BRANCH = "main"
LAB_DIR = Path("/content/day08-2A202603132")
if not LAB_DIR.exists():
    subprocess.run(
        ["git", "clone", "--quiet", "--branch", REPO_BRANCH, REPO_URL, str(LAB_DIR)],
        check=True,
    )
elif not (LAB_DIR / ".git").is_dir():
    raise RuntimeError(f"{LAB_DIR} đã tồn tại nhưng không phải repo Git. Hãy chọn thư mục khác.")

# Cập nhật đúng repo kể cả khi runtime còn bản clone từ repo đề.
# Chỉ fast-forward; Git sẽ dừng nếu có thay đổi xung đột, không xóa file người dùng.
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=LAB_DIR, check=True)
subprocess.run(["git", "fetch", "--quiet", "origin", REPO_BRANCH], cwd=LAB_DIR, check=True)
subprocess.run(["git", "checkout", "--quiet", REPO_BRANCH], cwd=LAB_DIR, check=True)
subprocess.run(["git", "merge", "--quiet", "--ff-only", f"origin/{REPO_BRANCH}"], cwd=LAB_DIR, check=True)
os.chdir(LAB_DIR)
print("Thư mục lab:", Path.cwd())
print("Repo:", REPO_URL, "| nhánh:", REPO_BRANCH)
GOLD_MANUAL_DONE = False


def require_manual_gold():
    global GOLD_MANUAL_DONE
    if not GOLD_MANUAL_DONE:
        confirmation = input(
            "Chỉ tiếp tục nếu bạn đã tự gán tay, lưu và export đủ 4 ảnh gold. "
            "Gõ DA_EXPORT_GOLD_2A202603132 để xác nhận: "
        ).strip()
        if confirmation != "DA_EXPORT_GOLD_2A202603132":
            raise RuntimeError("Chưa xác nhận hoàn tất gold. Dừng trước khi chạy AI.")
        GOLD_MANUAL_DONE = True


## 1. Cài thư viện (torch đã có sẵn trên Colab, không cài lại)

In [ ]:
!pip -q install "ultralytics>=8.4.0" opencv-python-headless pyyaml matplotlib "transformers>=5.0" huggingface_hub
!python scripts/check_env.py || true

## 2. Tải trọng số (bài chính: 3 model, ~0.35 GB)
`core` gồm YOLO26m, YOLOE-26s, **SAM 2.1 base**. SAM là bắt buộc: muốn SAM 1 bỏ comment dòng `--set sam1`.

In [ ]:
!python scripts/download_models.py --set core
# !python scripts/download_models.py --set sam1      # tuỳ chọn: SAM 1 (mobile_sam)

### (Nâng cao, tuỳ chọn) SAM 3
Cần quyền tại https://huggingface.co/facebook/sam3. Lưu token vào **Colab Secrets** (chìa khoá bên trái) tên `HF_TOKEN`, bật *Notebook access*. **Không dán token vào notebook.**

In [ ]:
# from google.colab import userdata
# import os
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# !python scripts/download_models.py --set sam3

## 3. Part 2 — Thử 3 model trên 8 ảnh `pilot` (không phải gold)
Mỗi lần chạy ghi một dòng vào `work/runs.csv`. Xem ảnh `viz/` bằng ô kế tiếp rồi ghi `model_log.csv` **trên máy bạn**.

In [ ]:
require_manual_gold()
!python scripts/prelabel.py --model yolo26 --size m --split pilot --conf 0.25 --out work/prelabel/t_y26
!python scripts/prelabel.py --model yolo26 --size m --split pilot --conf 0.5  --out work/prelabel/t_y26_c05
!python scripts/prelabel.py --model yoloe --classes three_wheeler --split pilot --conf 0.25 --out work/prelabel/t_yoloe
# SAM (bắt buộc): chọn phiên bản bằng --sam (mobile_sam.pt = SAM 1, sam2.1_b.pt = SAM 2.1)
!python scripts/prelabel.py --model yolo26 --size m --classes "person,motorbike,car" --images img_004.jpg img_013.jpg --conf 0.3 --segmenter sam --sam sam2.1_b.pt --out work/prelabel/t_sam

In [ ]:
from IPython.display import Image, display
display(Image("work/prelabel/t_y26/viz/img_011.jpg"))
display(Image("work/prelabel/t_yoloe/viz/img_002.jpg"))
display(Image("work/prelabel/t_sam/viz/img_013.jpg"))

## 4. Chạy mọi ảnh ở ngưỡng thấp (nguồn cho Part 3, 4)
Chạy **sau khi** bạn đã gán xong 4 ảnh gold bằng tay (Part 1). Ảnh gold không bao giờ được xem pre-label trước khi gán xong.

In [ ]:
require_manual_gold()
!python scripts/prelabel.py --model yolo26 --size m --split all --conf 0.05 --out work/prelabel/y26_all --no-viz
!python scripts/prelabel.py --model yoloe --classes three_wheeler --split all --conf 0.05 --out work/prelabel/yoloe_all --no-viz

## 5. Part 3 — tải kết quả thô về máy, quét ngưỡng ở máy bạn
Chạy ô dưới, tải `colab_stage1.zip`, giải nén vào repo trên máy, rồi chạy `threshold_sweep.py` (xem `docs/THRESHOLD.md`).

In [ ]:
!zip -rq colab_stage1.zip work/prelabel/y26_all work/prelabel/yoloe_all work/runs.csv
from google.colab import files
files.download("colab_stage1.zip")

## 6. Part 4 — gộp batch với ngưỡng của bạn
Sau Part 3 (`threshold_sweep.py` chạy **trên máy bạn**), dán chuỗi ngưỡng vào `THR`. Cài **ảnh bẫy** (`inject_traps.py`) nên làm trên máy bạn để hạt giống và đáp án nằm trong repo của bạn.

In [ ]:
require_manual_gold()
THR = ""  # Dán nguyên chuỗi ngưỡng từ submission/qc/threshold_sweep.md trên máy bạn.
if not THR.strip():
    raise ValueError("Chưa có ngưỡng Part 3. Dán kết quả threshold_sweep trước khi gộp.")
!python scripts/merge_prelabels.py work/prelabel/yoloe_all/pre.coco.json work/prelabel/y26_all/pre.coco.json --thr "$THR" --conf 0.3 --iou 0.6 --split "batch,gold" --out work/prelabel/merged_review/pre.coco.json
# Tạo bẫy trên máy local với --student "2A202603132"; chưa mở traps_key.json.


## 7. Part 5 — mask SAM cho 3 vật (bằng click)
Sửa toạ độ click theo vật bạn chọn; xem `docs/MODELS.md` mục B1.

In [ ]:
require_manual_gold()
SAM = "sam2.1_b.pt"     # SAM 1: mobile_sam.pt | sam_b.pt ; SAM 2.1: sam2.1_t/s/b/l.pt
!python scripts/sam2_points.py --image img_013.jpg --pos "400,480" --pos "600,110" --neg "720,300" --sam $SAM --out submission/masks --name tuktuk_img013
from IPython.display import Image, display
display(Image("submission/masks/tuktuk_img013.jpg"))

## 8. Tải kết quả về máy
Gói `work/prelabel` (file nhãn), `work/runs.csv` (nhật ký lần chạy, để copy sang `model_log.csv`) và `submission/masks`. Giải nén vào **repo trên máy bạn**, giữ nguyên đường dẫn.

In [ ]:
!zip -rq colab_outputs.zip work/prelabel work/runs.csv submission/masks 2>/dev/null || true
from google.colab import files
files.download("colab_outputs.zip")

**Tiếp theo (trên máy bạn):** giải nén `colab_outputs.zip` vào thư mục repo → làm theo `GUIDE.md` từ bước nạp vào CVAT. Nhớ: ghi `model_log.csv` bằng nhận xét **của bạn** khi nhìn ảnh `viz/`; số trong `runs.csv` do Colab chạy (ghi chú `device = cuda`, thời gian khác máy bạn).